# Coupled CPN + PINN Shape-Optimization Plots

This notebook is the first direct test of the geometry-optimization idea: can a Coordinate Projection Network deform the circular reference trap while a Vanilla PINN still represents an electrostatic potential that satisfies Laplace's equation?

The coupled model is `phi(z) = NN_theta(NN_phi(z))`. `NN_phi` maps reference coordinates to physical coordinates, and `NN_theta` predicts the RF potential on those mapped coordinates. The design pressure comes from the pseudopotential target: we ask the learned field to look like an ideal quadratic trap near the center.

Why this matters: a smooth potential plot alone is not enough. A shape optimizer is only useful if the deformation is physically valid, gradients reach both networks, the pseudopotential improves, and the map does not fold. Each plot below checks one of those failure modes.

Change `VARIATION` to `"A"` for PINN-first or `"C"` for joint training. The current markdown interpretation is written for the saved/current `C` joint run, which is the main stress test of unconstrained coupled optimization.

Project TODOs covered by this notebook: validate gradient flow through CPN and PINN, test whether unconstrained shape optimization is viable, diagnose map folding, evaluate pseudopotential-target matching, and decide which geometry constraints are needed next.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display

SPINN_ROOT = Path.cwd()
while SPINN_ROOT.name != "SPINN" and SPINN_ROOT != SPINN_ROOT.parent:
    SPINN_ROOT = SPINN_ROOT.parent
if str(SPINN_ROOT) not in sys.path:
    sys.path.insert(0, str(SPINN_ROOT))

from experiments.cpn_pinn_shape_optimization import figures

def show(fig):
    display(fig)
    plt.close(fig)

VARIATION = "C"
STEPS = 2000
run = figures.run_variation(variation=VARIATION, steps=STEPS, seed=0)
print("gradient flow:", run.result["gradient_flow"])
print("skipped non-finite steps:", run.result["skipped_steps"])


**Setup.** This cell trains one selected variation and builds the `RunArtifacts` object used by every later plot. It is not just boilerplate: it is the gradient-flow gate for the experiment.

What to check:

- `pinn_grads_finite_nonzero=True` means the potential network receives useful gradients.
- `cpn_grads_finite_nonzero=True` means the geometry map receives useful gradients through the PDE and pseudopotential losses.
- `skipped_nonfinite_steps` reveals numerical instability. Many skipped steps usually mean the map or pullback derivatives are blowing up.

For the joint `C` run, both parameter groups receive gradients and no steps are skipped, so the wiring is alive. That does **not** mean the geometry is valid; it only ticks off the TODO: *prove the coupled loss can differentiate through both networks without accidental detachments*. The later Jacobian and deformation plots decide whether those gradients produce a physical design.

## Physical potential

In [ ]:
show(figures.plot_potential(run))


**What the plot shows.** This is the learned RF potential `phi` on the physical coordinates produced by the CPN. The white circles are electrode masks. A physically plausible trap should interpolate smoothly between driven East/West electrodes and grounded North/South/world boundaries, with strong structure near the electrodes.

**How it helps.** This plot checks whether the coupled optimizer still learns an electrostatic-looking field after geometry deformation. It is a sanity check for the field solve, but it is not sufficient: a collapsed coordinate map can still produce a smooth-looking or nearly constant potential.

**Deduction from the current joint run.** The field is nearly flat compared with a real RF solution. That tells us the optimizer found a degenerate route: instead of creating a better electrode geometry, it weakened the field structure. This explains why the pseudopotential becomes tiny and why the next plots fail.

**TODO checked.** Evaluate whether the coupled CPN+PINN still solves a meaningful potential field. Current answer: *not yet; the unconstrained map allows a degenerate field solution*.
$∇^2V(x,y)=0(x,y)∈Ω$
$V=±V_0$ on RF electrodes
$V=0$ on outer boundary

## Model vs. ideal pseudopotential

In [ ]:
show(figures.plot_pseudopotential(run))


**What the plot shows.** The left panel is the model pseudopotential, computed from the electric field as `Psi_model proportional |grad_x phi|^2`. The right panel is the ideal target `Psi_ideal = k(x^2 + y^2)`, a quadratic bowl near the trap center.

**How it helps.** This is the actual design-objective plot. We are not merely fitting voltage; we want the RF pseudopotential to become more quadrupole-like. A successful shape optimizer should make the left panel resemble the right panel in the central fitting region.

**Deduction from the current joint run.** The model pseudopotential does not become a clean quadratic bowl. Its structure is dominated by a diagonal artifact from the distorted/folded coordinate map. The ideal target is smooth and radially quadratic, while the model is anisotropic and degenerate.

**TODO checked.** Test whether the learned geometry improves the ideal-pseudopotential objective. Current answer: *no; the optimizer exploits the unconstrained map instead of producing a physically meaningful quadrupole well*.

## Pseudopotential error

In [ ]:
show(figures.plot_pseudopotential_error(run))


**What the plot shows.** This is `Psi_model - Psi_ideal`. Near-zero error in the central region would mean the learned trap well matches the desired quadratic shape. Large coherent bands or lobes mean the learned pseudopotential has the wrong shape, not just the wrong scale.

**How it helps.** Error maps are more diagnostic than scalar loss: they reveal *where* and *how* the design objective fails. For shape optimization, spatial error structure can point back to the deformation field.

**Deduction from the current joint run.** The diagonal error band lines up with the deformation/Jacobian pathology. This is evidence that the pseudopotential failure is caused by the coordinate map, not just by an undertrained PINN head.

**TODO checked.** Localize pseudopotential-target mismatch and identify whether the failure is field-related or map-related. Current answer: *map-related, because the error geometry matches the folded deformation*.

## Jacobian determinant

In [ ]:
show(figures.plot_jacobian_determinant(run))


**What the plot shows.** The Jacobian determinant `det J` of the coordinate map `x = NN_phi(z)`. In 2D, `det J` measures local area scaling and orientation. A valid deformation should keep `det J > 0` everywhere, preferably not too close to zero. Negative values mean the map folded: two pieces of reference space overlap in physical space.

**How it helps.** This is the validity gate for coordinate projection. A shape result cannot be trusted if the map is non-invertible, even if the loss decreases or the potential looks smooth.

**Deduction from the current joint run.** The saved metrics report `det_jacobian_min = -26.99`, and the plot shows most of the domain at or below the invalid range. The map is not a deformation of the trap anymore; it is a folded coordinate transformation. That single fact invalidates the geometry as a candidate design.

**TODO checked.** Determine whether unconstrained CPN optimization preserves topology. Current answer: *no*. This directly motivates the next TODOs: Jacobian positivity, distortion bounds, smoothness, maximum displacement, and boundary anchoring.

## Electrode deformation

In [ ]:
show(figures.plot_deformation(run))


**What the plot shows.** Dashed gray curves are the original circular electrode boundaries. Red curves are where the CPN maps those same boundary points. A desirable result would show gentle, symmetric inward bowing toward a hyperbolic-like shape while remaining near the original trap scale.

**How it helps.** The Jacobian plot tells us whether the map is mathematically valid; this plot tells us whether the learned electrode geometry is physically interpretable. It answers the project-level question: are circles moving toward useful hyperbolic electrodes, or just distorting pathologically?

**Deduction from the current joint run.** The red boundaries are stretched into long diagonal slivers far outside the original electrode positions. That is not a hyperbolic-like optimization; it is a geometry collapse. In variation A, the CPN barely moves because the PINN-first/frozen setup is brittle. Together, A and C bracket the failure: one setup does not move, the other moves in a nonphysical way.

**TODO checked.** Inspect whether learned geometry moves toward a hyperbolic-like trap. Current answer: *no; unconstrained deformation is too free and needs physical/geometric regularization*.

## Loss curves

In [ ]:
show(figures.plot_loss_curves(run))


**What the plot shows.** Total, PDE, pseudopotential, and boundary losses during training. Loss curves tell us whether optimization is making steady progress, trading one objective against another, or entering an unstable regime.

**How it helps.** This plot connects the visual diagnostics to the optimizer. A low PDE loss alone is not success; Laplace can be satisfied by unhelpful fields. We need PDE, BC, pseudopotential, and map validity to improve together.

**Deduction from the current joint run.** PDE remains small and the boundary term is steady, while the pseudopotential term improves only after the geometry has already become invalid. The total loss does not certify a good design because the Jacobian/deformation plots show the optimizer used an illegal coordinate map to reduce parts of the objective.

**TODO checked.** Decide whether the first unconstrained coupled training run is scientifically usable. Current answer: *no, but it is useful as a failure-mode audit*. It tells us the next experiment should add constraints before claiming shape optimization: positive Jacobian, map smoothness, displacement bounds, electrode anchoring, symmetry, and gap preservation.

**Overall conclusion.** The experiment successfully tests the differentiable shape-optimization pipeline, and it proves gradients can reach both networks. It also proves the unconstrained version is not enough. That is a valuable result: the next milestone is not “more training,” but a constrained CPN objective that prevents folding while still allowing hyperbolic-like deformation.